<a href="https://colab.research.google.com/github/Thilac01/Hand_Written_Digit/blob/main/hand_written_digit_from_fully_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
file_location = r"/content/sample_data/mnist_train_small.csv"

# 1. Read data from CSV
with open(file_location, 'r') as f:
    raw_data = [[float(val) for val in line.strip().split(',')] for line in f if line.strip()]

# 2. Extract Labels and Features
y_raw = [int(row[0]) for row in raw_data]
X_raw = [row[1:] for row in raw_data]

# 3. Normalize Features to [0, 1]
# Pixel values range from 0 to 255
X_normalized = [[pixel / 255.0 for pixel in row] for row in X_raw]

# 4. One-Hot Encode Labels (10 classes: 0 through 9)
y_one_hot = []
for label in y_raw:
    one_hot = [0.0] * 10
    one_hot[label] = 1.0
    y_one_hot.append(one_hot)

print(f"Loaded dataset: {len(X_normalized)} samples, {len(X_normalized[0])} features per sample.")

In [ ]:
# Pure Python math implementations

def pure_exp(x):
    """Taylor series approximation for e^x."""
    # Clamp x to avoid numerical overflow/underflow
    if x > 20.0:
        x = 20.0
    elif x < -20.0:
        x = -20.0

    term = 1.0
    result = 1.0
    for i in range(1, 25):
        term *= x / i
        result += term
    return result

def pure_log(x):
    """Natural logarithm approximation using Newton-Raphson method."""
    if x <= 0:
        return -1e15
    y = 0.0  # Initial guess
    for _ in range(20):
        e_y = pure_exp(y)
        y = y + 2 * (x - e_y) / (x + e_y)
    return y

# Pseudo-Random Number Generator (Linear Congruential Generator)
prng_state = 123456789

def pure_random():
    """Generates a pseudo-random float in [0, 1)."""
    global prng_state
    prng_state = (1103515245 * prng_state + 12345) % 2147483648
    return prng_state / 2147483648.0

def pure_normal(mean=0.0, std=1.0):
    """Generates a standard normal random variable using Box-Muller transform."""
    u1 = pure_random()
    u2 = pure_random()
    while u1 <= 1e-15:
        u1 = pure_random()

    # Calculate sqrt(-2 * ln(u1)) using Taylor approximation for sqrt
    val = -2.0 * pure_log(u1)
    # Babylonians

In [ ]:
# Linear combination: Y = X * W + b
def linear_forward(X_sample, W, b):
    in_dim = len(X_sample)
    out_dim = len(b)
    Z = [0.0] * out_dim
    for j in range(out_dim):
        s = b[j]
        for i in range(in_dim):
            s += X_sample[i] * W[i][j]
        Z[j] = s
    return Z

# Activation Functions & Derivatives
def activate(Z, type_name='relu'):
    if type_name == 'relu':
        return [z if z > 0 else 0.0 for z in Z]
    elif type_name == 'tanh':
        res = []
        for z in Z:
            e2z = pure_exp(2.0 * z)
            res.append((e2z - 1.0) / (e2z + 1.0))
        return res
    elif type_name == 'sigmoid':
        return [1.0 / (1.0 + pure_exp(-z)) for z in Z]

def activation_derivative(A, type_name='relu'):
    if type_name == 'relu':
        return [1.0 if a > 0 else 0.0 for a in A]
    elif type_name == 'tanh':
        return [1.0 - a * a for a in A]
    elif type_name == 'sigmoid':
        return [a * (1.0 - a) for a in A]

def softmax(Z):
    max_z = max(Z)  # Numerical stability shift
    exp_Z = [pure_exp(z - max_z) for z in Z]
    sum_exp = sum(exp_Z)
    return [ez / sum_exp for ez in exp_Z]

print("Linear combination and activation functions defined.")

In [ ]:
class NeuralNetwork:
    def __init__(self, layer_sizes, activation='relu'):
        """
        layer_sizes: List containing [input_dim, hidden1, hidden2, ..., output_dim]
                     e.g., [784, 64, 32, 10] configures 2 hidden layers.
        activation: Hidden layer activation function ('relu' or 'tanh')
        """
        self.layer_sizes = layer_sizes
        self.activation = activation
        self.weights, self.biases = initialize_weights(layer_sizes)

    def forward(self, X_sample):
        activations = [X_sample]
        Z_list = []

        current_A = X_sample
        num_layers = len(self.weights)

        for l in range(num_layers):
            W = self.weights[l]
            b = self.biases[l]
            Z = linear_forward(current_A, W, b)
            Z_list.append(Z)

            if l == num_layers - 1:
                # Output layer uses Softmax activation
                current_A = softmax(Z)
            else:
                # Hidden layers use requested activation
                current_A = activate(Z, self.activation)
            activations.append(current_A)

        return activations, Z_list

    def train_step(self, X_sample, y_sample, learning_rate=0.01):
        # 1. Forward Pass
        activations, _ = self.forward(X_sample)
        num_layers = len(self.weights)

        # 2. Compute Output Error Delta (Softmax + Cross Entropy derivative)
        y_hat = activations[-1]
        delta = [y_hat[k] - y_sample[k] for k in range(len(y_sample))]

        # Backpropagation loop through layers
        for l in range(num_layers - 1, -1, -1):
            A_prev = activations[l]
            W = self.weights[l]
            b = self.biases[l]

            in_dim = len(A_prev)
            out_dim = len(b)

            # Compute gradients w.r.t weights and biases
            dW = [[0.0] * out_dim for _ in range(in_dim)]
            db = [0.0] * out_dim

            for j in range(out_dim):
                db[j] = delta[j]
                for i in range(in_dim):
                    dW[i][j] = A_prev[i] * delta[j]

            # Compute delta for the previous layer (if not input layer)
            if l > 0:
                next_delta = [0.0] * in_dim
                da = activation_derivative(A_prev, self.activation)
                for i in range(in_dim):
                    error_sum = 0.0
                    for j in range(out_dim):
                        error_sum += W[i][j] * delta[j]
                    next_delta[i] = error_sum * da[i]
                delta = next_delta

            # Stochastic Gradient Descent Parameter Update
            for i in range(in_dim):
                for j in range(out_dim):
                    self.weights[l][i][j] -= learning_rate * dW[i][j]
            for j in range(out_dim):
                self.biases[l][j] -= learning_rate * db[j]

    def predict(self, X_sample):
        activations, _ = self.forward(X_sample)
        probs = activations[-1]
        # Argmax
        max_idx = 0
        max_val = probs[0]
        for i in range(1, len(probs)):
            if probs[i] > max_val:
                max_val = probs[i]
                max_idx = i
        return max_idx

print("Neural Network class instantiated successfully.")

In [ ]:
# Architecture Configuration:
# Input Layer: 784 features
# Hidden Layer 1: 64 neurons
# Hidden Layer 2: 32 neurons
# Output Layer: 10 neurons
# Activation: 'relu' (or switch to 'tanh')
layer_architecture = [784, 64, 32, 10]
activation_function = 'relu'

nn = NeuralNetwork(layer_architecture, activation=activation_function)

# Hyperparameters
epochs = 2
learning_rate = 0.005
num_samples_to_train = 1000  # Subset size for quick demonstration in pure Python

print(f"Training network {layer_architecture} with '{activation_function}' activation...")

for epoch in range(epochs):
    correct_predictions = 0
    total_loss = 0.0

    for i in range(num_samples_to_train):
        X_i = X_normalized[i]
        y_i = y_one_hot[i]

        # Train on single sample
        nn.train_step(X_i, y_i, learning_rate=learning_rate)

        # Evaluate accuracy & loss tracking
        pred = nn.predict(X_i)
        actual = y_raw[i]
        if pred == actual:
            correct_predictions += 1

        probs, _ = nn.forward(X_i)
        output_probs = probs[-1]
        total_loss -= pure_log(output_probs[actual] + 1e-12)

    acc = (correct_predictions / num_samples_to_train) * 100
    avg_loss = total_loss / num_samples_to_train
    print(f"Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.4f} | Accuracy: {acc:.2f}%")

# Test prediction on a sample
sample_index = 0
prediction = nn.predict(X_normalized[sample_index])
actual_label = y_raw[sample_index]

print("\n--- Inference Test ---")
print(f"Sample Index: {sample_index}")
print(f"Actual Digit: {actual_label}")
print(f"Model Predicted Digit: {prediction}")